In [ ]:
%matplotlib inline
from pathlib import Path
import math
import os
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a test run, a few training steps; grading always uses it.
# False: the full training run (about a minute on a laptop CPU).
IS_UNITTEST = False
# Training steps per epoch and epochs in the full run.
STEPS_PER_EPOCH = 200
EPOCHS = 50
# Rays per optimizer step.
BATCH_SIZE = 1024
# Points sampled along each ray.
N_SAMPLES = 32
# Width of the hidden layers and number of sine/cosine frequency bands.
HIDDEN_SIZE = 128
N_FREQUENCIES = 6
# Adam learning rate.
LEARNING_RATE = 5.0e-3
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)


seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
src = source.code.src
nerf_data = src.datasets.nerf
nerf_models = src.models.feedforward.nerf
answer = {}

# Tiny NeRF: a 3D Scene from Photos

NeRF represents a scene as a function from a **3D point** to its density
and color. To render a pixel, we cast a ray from the camera through the
pixel, sample points along it, ask the network for density and color at
every point, and blend the colors according to the densities. Training
compares the rendered pixels with the photos.

We use the *mic* scene of the [NeRF synthetic dataset](https://github.com/bmild/nerf)
(a studio microphone; model by up3d.de, public domain, CC-0): 106 rendered
photos with known camera poses, at 50 × 50 pixels. 100 photos are for
training; 6 are held out, and at the end we render them from their camera
poses. The test run (`IS_UNITTEST = True`) uses the same data and settings
and only makes a few training steps.

**Files to edit:**
- [src/datasets/nerf.py](../../source/code/src/datasets/nerf.py): the camera rays;
- [src/models/feedforward/nerf.py](../../source/code/src/models/feedforward/nerf.py): the network, the positional encoding and the rendering;
- the marked cells of this notebook: the seeds and the loss.


# Task 0: Prepare the Environment

## Goal

Set all random seeds with `src.utils.seed.seed_all(0)`. NeRF samples the points along every ray at random, so the seeds decide the result.


In [ ]:
# Edit this cell in task.ipynb.
## YOUR CODE HERE
seed_check = {
    'python': [random.random() for _ in range(3)],
    'numpy': np.random.rand(2),
    'torch': torch.rand(2),
}
answer['seed_check'] = seed_check
pprint(seed_check)


In [ ]:
course_utils.checker.print_progress(
    'Task 0: Prepare the Environment', answer,
    prefixes=('seed_check',),
    is_small_run=IS_UNITTEST)


# The Data

`src.datasets.nerf.load_scene` reads the photos and returns a dict:

- `'train_images'`: `[100, 50, 50, 3]`, RGB in [0, 1];
- `'heldout_images'`: `[6, 50, 50, 3]`;
- `'train_poses'`, `'heldout_poses'`: one 4 × 4 camera-to-world matrix per
  photo. Its top-left 3 × 3 block turns directions from camera space to
  world space; its last column is the camera position;
- `'focal'`: the focal length in pixels.


In [ ]:
scene = nerf_data.load_scene('mic_50.npz')
print('training photos:', tuple(scene['train_images'].shape),
      'held-out photos:', tuple(scene['heldout_images'].shape), 'focal:', scene['focal'])

fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, view in zip(axes, range(0, 100, 20)):
    ax.imshow(scene['train_images'][view])
    ax.set_title(f'photo {view}')
    ax.axis('off')
plt.tight_layout()
plt.show()


# Task 1: Cast Camera Rays

## Goal

Implement the ray computation in `NeRFRayDataset.__init__` in
[src/datasets/nerf.py](../../source/code/src/datasets/nerf.py). The dataset has one item per pixel
of every photo: the ray from the camera through the pixel and the pixel's
color. `__len__` and `__getitem__` are given; `__getitem__` also accepts a
list of indices and then returns a whole batch.

## The Camera

The camera looks along its own $-z$ axis; $x$ points right, $y$ up. The
pixel in row $r$ and column $c$ of an $H \times W$ photo is seen in the
camera-space direction

$$d_{cam} = \left(\frac{c - W/2}{f},\; -\frac{r - H/2}{f},\; -1\right),$$

where $f$ is the focal length. With the pose's rotation $R$ (top-left
3 × 3) and the camera position $t$ (last column):

- direction: $d = R\, d_{cam}$, divided by its length;
- origin: $t$, the same for every pixel of the photo.

Compute the rays of all pixels of all photos at once, with tensors of
shape `[views, height, width, 3]`; `torch.meshgrid` gives the rows and the
columns of every pixel.


In [ ]:
# Edit: ../../source/code/src/datasets/nerf.py
train_dataset = nerf_data.NeRFRayDataset(scene['train_images'], scene['train_poses'], scene['focal'])
heldout_dataset = nerf_data.NeRFRayDataset(scene['heldout_images'], scene['heldout_poses'], scene['focal'])
# ^^^ This class should be implemented in this task

print('training rays:', len(train_dataset), 'held-out rays:', len(heldout_dataset))
# The middle pixel of the first photo and a corner pixel.
rays = train_dataset[[25 * 50 + 25, 0]]
pprint(rays)
answer['rays'] = rays


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Cast Camera Rays', answer,
    prefixes=('rays',),
    is_small_run=IS_UNITTEST)


# Task 2: Encode Positions and Predict the Field

## Goal

Implement two parts of `TinyNeRF` in [src/models/feedforward/nerf.py](../../source/code/src/models/feedforward/nerf.py):

- `positional_encoding`: an MLP on raw $(x, y, z)$ can only draw smooth
  shapes. Add $\sin(2^k \pi x)$ and $\cos(2^k \pi x)$ of each coordinate
  for $k = 0, \dots, K - 1$, so that the network can also follow fast
  changes. With $K$ = `n_frequencies`, a point becomes $3 (1 + 2K)$ numbers.
- `__init__`: `self.field` = `Linear → ReLU → Linear → ReLU`, `hidden_size`
  wide, and the two heads that read it: `self.density_head` (1 output) and
  `self.color_head` (3 outputs).

`query_field` is given: it makes the density non-negative with `softplus`
and the color lie in [0, 1] with `sigmoid`.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/nerf.py
model = nerf_models.TinyNeRF(hidden_size=HIDDEN_SIZE, n_frequencies=N_FREQUENCIES,
                             n_samples=N_SAMPLES)
# ^^^ This class should be implemented in this task

check_model = nerf_models.TinyNeRF(hidden_size=HIDDEN_SIZE, n_frequencies=N_FREQUENCIES,
                                   n_samples=N_SAMPLES)
# Predictable weights, so the values below can be compared with the answer.
src.utils.deterministic_init(check_model)
check_points = torch.tensor([[0.0, 0.0, 0.0], [0.5, -0.25, 1.0], [-1.0, 0.75, 0.5]])
with torch.no_grad():
    encoded = check_model.positional_encoding(check_points)
    density, color = check_model.query_field(check_points)
print('encoded points:', tuple(encoded.shape))
answer['field'] = {'encoded': encoded, 'density': density, 'color': color}


In [ ]:
course_utils.checker.print_progress(
    'Task 2: Encode Positions and Predict the Field', answer,
    prefixes=('field',),
    is_small_run=IS_UNITTEST)


# Task 3: Render the Rays

## Goal

Implement the compositing in `TinyNeRF.render` in [src/models/feedforward/nerf.py](../../source/code/src/models/feedforward/nerf.py).
Along a ray, sample $i$ has density $\sigma_i$, color $c_i$ and step length
$\delta_i$ to the next sample. The sample is opaque with probability

$$\alpha_i = 1 - e^{-\sigma_i \delta_i},$$

and light reaches it only if all the samples before it let it pass:

$$T_i = \prod_{j < i} (1 - \alpha_j), \qquad w_i = T_i \alpha_i, \qquad \mathrm{rgb} = \sum_i w_i c_i.$$

Use `torch.cumprod` for $T_i$ (shift it by one sample, so that $T_0 = 1$).
The background is black, so nothing is added for light that passes all
samples.

`forward` is given: it places `n_samples` points on each ray between
`near` and `far` (at random inside equal bins while training, in the bin
middles otherwise), renders them, and writes `batch['signals']['rgb']`,
`['weights']` and `['depth']` (the weighted mean depth).


In [ ]:
# Edit: ../../source/code/src/models/feedforward/nerf.py
check_model.eval()
with torch.no_grad():
    check_batch = check_model({'data': heldout_dataset[[25 * 50 + 25, 0]]})
pprint(check_batch['signals'])
answer['render'] = check_batch['signals']


In [ ]:
course_utils.checker.print_progress(
    'Task 3: Render the Rays', answer,
    prefixes=('render',),
    is_small_run=IS_UNITTEST)


# Task 4: Create the Loss

## Goal

Complete `rgb_mse` below: the mean squared error between the rendered
colors `batch['signals']['rgb']` and the photo's colors
`batch['data']['target_rgb']`:

$$L = \frac{1}{3N} \sum_{i=1}^{N} \lVert \mathrm{rgb}_i - \mathrm{target}_i \rVert^2.$$

Image quality is usually given as PSNR, $-10 \log_{10} L$ in decibels:
the higher, the better.


In [ ]:
# Edit this cell in task.ipynb.
def rgb_mse(batch):
    return batch['signals']['rgb'].abs().mean()

## YOUR CODE HERE

losses = {'mse': rgb_mse}

scored_batch = {'signals': {'rgb': torch.tensor([[0.0, 0.5, 1.0], [0.2, 0.2, 0.2]])},
                'data': {'target_rgb': torch.tensor([[0.0, 0.0, 1.0], [0.0, 0.2, 0.4]])}}
answer['test_loss_val'] = {name: fn(scored_batch) for name, fn in losses.items()}
pprint(answer['test_loss_val'])


In [ ]:
course_utils.checker.print_progress(
    'Task 4: Create the Loss', answer,
    prefixes=('test_loss_val',),
    is_small_run=IS_UNITTEST)


# Training

`src.train_loop.train_model`, the training loop from the Shallow
Feedforward NNs task, trains the field. Every step takes `BATCH_SIZE`
random rays from all training photos; an epoch is `STEPS_PER_EPOCH` steps.
After every epoch it renders the held-out photos and reports the loss on
them.


In [ ]:
if IS_UNITTEST:
    # The test run: the same data and settings, just a few steps.
    EPOCHS, STEPS_PER_EPOCH = 2, 10

# Each item of these loaders is a whole batch: the sampler hands the
# dataset a list of ray indices.
ray_sampler = torch.utils.data.RandomSampler(
    train_dataset, replacement=True, num_samples=STEPS_PER_EPOCH * BATCH_SIZE,
    generator=torch.Generator().manual_seed(seed))
train_dl = DataLoader(train_dataset, batch_size=None,
                      sampler=torch.utils.data.BatchSampler(ray_sampler, BATCH_SIZE, drop_last=True))
heldout_dl = DataLoader(heldout_dataset, batch_size=None,
                        sampler=torch.utils.data.BatchSampler(
                            torch.utils.data.SequentialSampler(heldout_dataset), 2500, drop_last=False))

src.utils.seed.seed_all(0)
model = nerf_models.TinyNeRF(hidden_size=HIDDEN_SIZE, n_frequencies=N_FREQUENCIES,
                             n_samples=N_SAMPLES)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = src.train_loop.train_model(model, EPOCHS, train_dl, heldout_dl, losses, optimizer,
                                     run_name='tiny nerf')

# max(): an untrained model can have an error of exactly 0 (all black on black).
heldout_psnr = [-10 * math.log10(max(loss, 1.0e-10)) for loss in history['valid_loss']['mse']]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
epochs = range(1, EPOCHS + 1)
axes[0].plot(epochs, history['train_loss']['mse'], label='training')
axes[0].plot(epochs, history['valid_loss']['mse'], label='held-out')
axes[0].set_title('RGB mean squared error')
axes[0].legend()
axes[1].plot(epochs, heldout_psnr)
axes[1].set_title('held-out PSNR, dB')
for ax in axes:
    ax.set_xlabel('epoch')
plt.show()
print(f'held-out PSNR after training: {heldout_psnr[-1]:.1f} dB')

answer['training'] = {'train_mse': history['train_loss']['mse'],
                      'heldout_mse': history['valid_loss']['mse']}


In [ ]:
course_utils.checker.print_progress(
    'Training', answer,
    prefixes=('training',),
    is_small_run=IS_UNITTEST)


# What the Model Learned

The six held-out photos were never used for training. We render each of
them from its camera pose: the photo, the rendering, and the depth the
model assigns to every pixel (bright = far).


In [ ]:
model.eval()
with torch.no_grad():
    rendered = model({'data': heldout_dataset[list(range(len(heldout_dataset)))]})
n_views = len(scene['heldout_images'])
images = rendered['postprocessed']['rgb'].reshape(n_views, 50, 50, 3)
depths = rendered['signals']['depth'].reshape(n_views, 50, 50)

fig, axes = plt.subplots(3, n_views, figsize=(2 * n_views, 6.3))
for view in range(n_views):
    axes[0, view].imshow(scene['heldout_images'][view])
    axes[1, view].imshow(images[view])
    axes[2, view].imshow(depths[view], cmap='magma')
for row, title in enumerate(['photo', 'rendered', 'depth']):
    axes[row, 0].set_ylabel(title)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

answer['heldout_render'] = {'mean_rgb': images.mean(dim=(1, 2)), 'mean_depth': depths.mean(dim=(1, 2))}


In [ ]:
course_utils.checker.print_progress(
    'What the Model Learned', answer,
    prefixes=('heldout_render',),
    is_small_run=IS_UNITTEST)


# Improving the Result

Set `IS_UNITTEST = False` for the full training. Then try more `EPOCHS`,
more `N_SAMPLES` along each ray, a wider field (`HIDDEN_SIZE`), or a
different number of frequencies (`N_FREQUENCIES`), and watch the held-out
PSNR. With `N_FREQUENCIES = 0` the field gets only the raw coordinates:
compare how sharp the renderings are.


In [ ]:
course_utils.checker.save_answer(answer)
